# Deliverable 2: modelli originali, implementazione corretta per Colab

## In breve
Questo notebook conserva le likelihood e i priori di **A, B, C, D1 e D2** di
`coursework/deliverable2_v0`. Corregge la preparazione dei dati, i confronti e la logica
dei verdetti. Il notebook di partenza non viene modificato. Nessun risultato empirico
è anticipato: le tabelle finali vengono costruite solo dagli output del run corrente.

**Avvio:** caricare il notebook su Colab ed eseguire le celle dall'inizio. Non serve
scegliere una `DATA_DIR`: con `COLLECT_FROM_SCRATCH=True` il notebook clona la revisione
dichiarata, esegue un preflight della raccolta e ricrea tutte le osservazioni in una nuova
cartella con manifest e hash. La modalità `preflight` disattiva solo MCMC, non la raccolta
fresca. Dopo la prima raccolta, impostare `COLLECT_FROM_SCRATCH=False` per
riutilizzare esclusivamente quella registrata dal notebook, senza indicare percorsi. `smoke` prova la procedura su campioni piccoli, sempre NON RIPORTABILI. `full` usa
la raccolta completa per i fit principali e attiva i controlli costosi, che possono durare ore.

## Contesto e metodi
Le quindici geometrie sono quelle fittabili al contesto di 480 campioni del Deliverable 2.
Il run p32-s28 è escluso perché il suo stride non divide il contesto.

| Correzione | Scelta esplicita |
|---|---|
| Sfondo di A/C | Una categoria per `(generator, bg_id)` |
| Fase di C | Otto intervalli uniformi di `phase mod 2π`, non `phase_idx` |
| Contrasti | Triplette complete, finite e con recovery non negative; nessun filtro sul risultato |
| Vecchio `live` | Registrato per audit, non usato per cambiare la popolazione |
| M1 | `Pr(delta_O > 0)`: erratum al segno negativo della tabella v0 |
| B | Soglie 20%/10% aggiunte esplicitamente: non erano nella tabella v0 |
| H2 | ROPE e LOO separati: non è richiesta la vittoria del modello con fase |
| D1 | Un verdetto congiunto, probabilità congiunta dei due segni, tutti i gate applicati |
| D2 | Primo sito e fondamentale distinti; la selezione teorica non convalida il movimento |
| Cache | Hash di codice, dati, selezioni, priori, backend e campionamento; scrittura atomica |

**Limite dei dati D2.** `02_sites.parquet` contiene siti già attribuiti mediante le griglie
teoriche. Il primo sito può essere un'armonica superiore e la vicinanza alla griglia è
una condizione di selezione. Da questi dati si produce una diagnostica condizionale,
non una conferma indipendente del movimento. Per un verdetto D2 occorre una tabella di
fondamentali misurati indipendentemente, con il contratto documentato nella sezione D2.
La mancanza di questa evidenza resta visibile: non viene sostituita da un `True`.

Le scelte nuove di validazione (copertura PPC, stabilità delle decisioni e numerosità minima)
sono controlli operativi di questo notebook, non soglie attribuite retroattivamente al testo.
Convergenza, recovery sintetico e adeguatezza ai dati sono evidenze distinte.

In [ ]:
#@title 1. Dipendenze Colab (nessuna installazione fuori Colab)
import sys, os, subprocess, importlib.metadata as metadata
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
INSTALL_NUMPYRO = False  #@param {type:'boolean'}
REQUIREMENTS = ['pymc==5.28.5', 'arviz==0.22.0', 'numpy>=1.26,<3',
                'pandas>=2,<4', 'scipy>=1.11,<2', 'pyarrow', 'h5netcdf', 'h5py',
                'matplotlib', 'packaging']
if IN_COLAB:
    from packaging.requirements import Requirement
    missing = []
    for text in REQUIREMENTS + (['numpyro'] if INSTALL_NUMPYRO else []):
        requirement = Requirement(text)
        try:
            version = metadata.version(requirement.name)
            if version not in requirement.specifier:
                missing.append(text)
        except metadata.PackageNotFoundError:
            missing.append(text)
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS,
                        *(['numpyro'] if INSTALL_NUMPYRO else [])], check=True)
        raise RuntimeError('Dipendenze installate. Riavviare il runtime Colab e rieseguire dall’inizio.')
print('Preparazione dipendenze terminata. Nessun campionamento eseguito.')


In [ ]:
#@title 2. Configurazione: preflight, smoke oppure full
MODE = 'preflight'  #@param ['preflight', 'smoke', 'full']
DATA_DIR = ''  # assegnata dalla raccolta fresca; non impostare manualmente
OUT_DIR = ''  #@param {type:'string'}
D2_MEASUREMENTS = ''  #@param {type:'string'}
MOUNT_DRIVE = True  #@param {type:'boolean'}
COLLECT_FROM_SCRATCH = True  #@param {type:'boolean'}
COLLECTION_DEVICE = 'auto'  #@param ['auto', 'cpu', 'cuda']
COLLECTION_BATCH_SIZE = 8  #@param {type:'integer'}
REPO_URL = 'https://github.com/FedericoSabbadini/patchAliasing.git'
REPO_REF = 'main'  #@param {type:'string'}
ENGINE = 'auto'  #@param ['auto', 'pymc', 'numpyro']
NON_CENTERED = True  #@param {type:'boolean'}
DENSE_MASS = False  #@param {type:'boolean'}
SEED = 42
FS, BAND, EPS, N_PHASE = 512.0, (2.0, 250.0), 0.01, 8
POPULATION = [(8,8),(16,8),(16,12),(16,16),(24,8),(24,12),(24,16),(24,20),
              (24,24),(32,8),(32,12),(32,16),(32,20),(32,24),(32,32)]
TAGS = {f'p{p}-s{s}' for p,s in POPULATION}
RHAT_MAX, ESS_MIN, PROB = 1.01, 1000, 0.95
LOG08, LOG11, LOG12 = __import__('math').log(.8), __import__('math').log(1.1), __import__('math').log(1.2)
GRID_TOL, DELTA_F, MIN_SITES = 5e-6, 1.0, 10
CHAINS, BLOCK_CHAINS = 4, 2
DRAWS, TUNE = (1500, 1500) if MODE == 'full' else (60, 60)
VALID_DRAWS = 1500 if MODE == 'full' else 60
LOO_ROWS = 20000 if MODE == 'full' else 240
SMOKE_ROWS = 300
PPC_DRAWS = 300 if MODE == 'full' else 24
PPC_BATCH_DRAWS = 8
PRIOR_DRAWS = 200 if MODE == 'full' else 24
RECOVERY_REPEATS = 3 if MODE == 'full' else 1
assert MODE in {'preflight', 'smoke', 'full'}
if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
if not OUT_DIR:
    drive_root = Path('/content/drive/MyDrive/patchAliasing')
    OUT_DIR = str((drive_root if drive_root.parent.is_dir() else Path.cwd()) / '_run_D2_corretto')
RUN_ROOT = Path(OUT_DIR).expanduser().resolve()
COLLECTION_STATE = RUN_ROOT/'fresh_collection_state.json'
OUT = RUN_ROOT / MODE
OUT.mkdir(parents=True, exist_ok=True)
print(f'Modalità: {MODE}; risultati: {OUT}')
print('SMOKE: solo prova tecnica, nessun verdetto empirico.' if MODE == 'smoke'
      else 'FULL: fit principali e controlli costosi.' if MODE == 'full' else 'PREFLIGHT: nessun MCMC.')

In [ ]:
#@title 3. Import e funzioni comuni
import ast, copy, gc, hashlib, inspect, json, math, time, uuid
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats
CODE_SHA256 = '4c7240cc29a882c2b88c7e18e590dde62dda38dba06ee682aa3d843f6cef5b00'
VERSIONS = {n: metadata.version(n) for n in ['pymc','arviz','numpy','pandas','scipy','pytensor','xarray']}
print(VERSIONS)
plt.rcParams.update({'figure.dpi': 115, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .18, 'font.size': 10})

def show(frame):
    print(frame.to_string(index=False))

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, allow_nan=False).encode()).hexdigest()

def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str, allow_nan=False), encoding='utf8')
    os.replace(tmp, path)

def codes(frame, columns):
    values = pd.MultiIndex.from_frame(frame[list(columns)])
    labels, levels = pd.factorize(values, sort=True)
    if (labels < 0).any():
        raise ValueError(f'Indice mancante: {columns}')
    return labels.astype('int32'), len(levels)

def sample_balanced(frame, limit, strata, seed=SEED):
    # Una quota per ogni strato, poi riempimento casuale senza rimpiazzo.
    if len(frame) <= limit:
        return np.arange(len(frame))
    rng = np.random.default_rng(seed)
    groups = list(frame.groupby(list(strata), observed=True, dropna=False, sort=True).indices.values())
    if len(groups) > limit:
        raise ValueError(f'{len(groups)} strati richiedono almeno altrettante righe, limite={limit}')
    quota = max(1, limit // len(groups))
    chosen = np.concatenate([rng.choice(g, min(len(g), quota), replace=False) for g in groups])
    rest = np.setdiff1d(np.arange(len(frame)), chosen)
    if len(chosen) < limit:
        chosen = np.r_[chosen, rng.choice(rest, limit-len(chosen), replace=False)]
    return np.sort(chosen)

def canonical_contrasts(raw):
    """Stessa popolazione e stessa risposta per input largo o lungo; nessun filtro su live."""
    required = ['model','generator','bg_id','f_lock','phase_idx','phase','P','S','overlap']
    missing = sorted(set(required) - set(raw))
    if missing:
        raise ValueError(f'Contrasti: colonne indispensabili assenti: {missing}')
    d = raw[raw.model.isin(TAGS)].copy()
    keys = ['model','generator','bg_id','f_lock','phase_idx']
    if d[required].isna().any().any():
        raise ValueError('Metadati dei contrasti mancanti: non si possono ricostruire le triplette.')
    if not np.isfinite(d[['phase','f_lock','P','S','overlap']].to_numpy(float)).all():
        raise ValueError('Metadati numerici dei contrasti non finiti.')
    if {'R_lock','R_lo','R_hi'} <= set(d):
        if d.duplicated(keys).any():
            raise ValueError('Triplette duplicate nella forma larga.')
    elif 'R' in d:
        if 'role' in d:
            role = d.role.astype(str).str.lower().map({'lock':'lock','lo':'lo','hi':'hi',
                'low':'lo','high':'hi','minus':'lo','plus':'hi'})
            if role.isna().any():
                raise ValueError('Ruoli non riconosciuti: attesi lock/lo/hi.')
            d['_role'] = role
        elif 'is_lock' in d and 'f' in d:
            if not d.is_lock.isin([0,1,False,True]).all():
                raise ValueError('is_lock deve essere binario.')
            d['_role'] = np.where(d.is_lock.astype(bool), 'lock', np.where(d.f < d.f_lock,'lo','hi'))
        else:
            raise ValueError('Formato lungo senza role oppure is_lock e frequenza f.')
        if d.duplicated(keys+['_role']).any():
            raise ValueError('Bracci duplicati: vietato mediare silenziosamente misure diverse.')
        invariant = d.groupby(keys, observed=True)[['phase','P','S','overlap']].nunique(dropna=False)
        if (invariant > 1).any().any():
            raise ValueError('I bracci non condividono fase/geometria: tripletta non appaiata.')
        if 'f' in d:
            valid_f = ((d._role.eq('lock') & np.isclose(d.f, d.f_lock, atol=GRID_TOL, rtol=0)) |
                       (d._role.eq('lo') & (d.f < d.f_lock)) | (d._role.eq('hi') & (d.f > d.f_lock)))
            if not valid_f.all():
                raise ValueError('Frequenze dei bracci incompatibili con i ruoli.')
        values = d.pivot(index=keys, columns='_role', values='R').reindex(columns=['lock','lo','hi'])
        meta = d.groupby(keys, observed=True)[['phase','P','S','overlap']].first()
        d = meta.join(values.rename(columns={'lock':'R_lock','lo':'R_lo','hi':'R_hi'})).reset_index()
    else:
        raise ValueError('Schema dei contrasti non riconosciuto.')
    r = d[['R_lock','R_lo','R_hi']].to_numpy(float)
    valid = np.isfinite(r).all(1) & (r >= 0).all(1)
    calculated = np.full(len(d), np.nan)
    calculated[valid] = np.log(r[valid,0]+EPS) - .5*(np.log(r[valid,1]+EPS)+np.log(r[valid,2]+EPS))
    if 'd' in d and not np.allclose(d.loc[valid,'d'], calculated[valid], rtol=1e-7, atol=1e-8):
        raise ValueError('d salvato non coincide con Eq. d2contrast; correggere la provenienza dei dati.')
    audit = {'triplette':len(d), 'utilizzabili':int(valid.sum()), 'escluse_non_valide':int((~valid).sum()),
             'legacy_live':int(d.live.sum()) if 'live' in d else None,
             'regola':'tutte le triplette complete, finite e non negative; live ignorato'}
    d = d.loc[valid].copy()
    d['d'] = calculated[valid]
    angle = np.mod(d.phase.to_numpy(float), 2*np.pi)
    angle[np.isclose(angle, 2*np.pi, atol=1e-10, rtol=0)] = 0.0
    d['phase_angle'] = angle
    d['phase_bin'] = np.floor(angle/(2*np.pi/N_PHASE)).astype(int).clip(0, N_PHASE-1)
    d = d.sort_values(keys).reset_index(drop=True)
    return d, audit

In [ ]:
#@title 4. Raccolta facoltativa, con preflight obbligatorio e nessuna cancellazione
# Il driver conserva l'elaborazione per blocchi di siti della versione originale.
LOWMEM_DRIVER = '\nimport os, sys, gc, resource\nimport numpy as np, pandas as pd\nimport probe_lib as pl\nimport collect as C\n\nCHUNK = int(os.environ.get("SITI_PER_BLOCCO", "4"))\nCOLS = ["model","P","S","overlap","generator","bg_id","f_lock","family","cpp","delta",\n        "phase_idx","phase","role","f","is_lock","R","dphase","f_hat","h","h_truth"]\n\ndef _rss():\n    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024**2\n\ndef collect_contrasts_lowmem(probe, cfg):\n    P, S = probe.P, probe.S\n    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}\n    sites = [f for f, d in offsets.items() if np.isfinite(d)]\n    print("    siti utili: %d, a blocchi di %d" % (len(sites), CHUNK), flush=True)\n    frames = []\n    for gen in cfg.generators:\n        pool = pl.background_pool(gen, cfg.n_bg, pl.CTX + pl.PRED)\n        for c0 in range(0, len(sites), CHUNK):\n            blocco = sites[c0:c0+CHUNK]\n            contexts, futures, freqs, meta = [], [], [], []\n            for fk in blocco:\n                phases = pl.phases_Sf(fk, cfg.n_phase_contrast)\n                for bg_id, bg in enumerate(pool):\n                    for ph_idx, ph in enumerate(phases):\n                        d_fk = offsets[fk]\n                        for role, f in (("lock", fk), ("lo", fk - d_fk), ("hi", fk + d_fk)):\n                            full = pl.build_context(bg, f, ph, pl.CTX + pl.PRED)\n                            contexts.append(np.array(full[:pl.CTX]))\n                            futures.append(np.array(full[pl.CTX:]))\n                            freqs.append(f)\n                            meta.append(dict(generator=gen, bg_id=bg_id, f_lock=fk, delta=d_fk,\n                                             phase_idx=ph_idx, phase=float(ph), role=role,\n                                             f=float(f)))\n            if not contexts:\n                continue\n            R, dphase, f_hat, f_hat_truth = probe.measure(\n                np.stack(contexts), np.stack(futures), np.array(freqs), k=cfg.fhat_topk)\n            out = pd.DataFrame(meta)\n            out["R"] = R\n            out["dphase"] = dphase\n            out["f_hat"] = f_hat[:, 0]\n            out["h"] = pl.localisation_hit(f_hat, out["f"].to_numpy(float), tol=cfg.fhat_tol_hz)\n            out["h_truth"] = pl.localisation_hit(f_hat_truth, out["f"].to_numpy(float),\n                                                 tol=cfg.fhat_tol_hz)\n            frames.append(out)\n            del contexts, futures, freqs, meta, R, dphase, f_hat, f_hat_truth\n            gc.collect()\n            print("    %s siti %d-%d/%d  righe finora %d  RSS max %.1f GB"\n                  % (gen, c0+1, c0+len(blocco), len(sites),\n                     sum(len(x) for x in frames), _rss()), flush=True)\n    if not frames:\n        return pd.DataFrame()\n    out = pd.concat(frames, ignore_index=True)\n    out["is_lock"] = (out["role"] == "lock").astype(np.int8)\n    out["model"] = probe.tag\n    out["P"], out["S"] = P, S\n    out["overlap"] = (P - S) / P\n    out["cpp"] = out["f_lock"] * P / pl.FS\n    out["family"] = [pl.lock_family(f, P, S) for f in out["f_lock"]]\n    return out[COLS]\n\nC.collect_contrasts = collect_contrasts_lowmem\nprint("driver a memoria limitata attivo (blocchi da %d siti)" % CHUNK, flush=True)\nsys.exit(C.main(sys.argv[1:]))\n'

if COLLECT_FROM_SCRATCH:
    if not IN_COLAB:
        raise RuntimeError('La raccolta fresca richiede Colab/Linux; DATA_DIR non viene cercata automaticamente.')
    clone = Path('/content/patchAliasing_D2_corretto')
    if not clone.exists():
        subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(clone)], check=True)
    commit = subprocess.check_output(['git','-C',str(clone),'rev-parse','HEAD'], text=True).strip()
    pipe = clone/'chronos/bayesian/support_scripts'
    for name in ['collect.py','probe_lib.py','checkpointing.py','model_loader.py']:
        if not (pipe/name).is_file():
            raise FileNotFoundError(pipe/name)
    import tomllib
    deps = tomllib.loads((clone/'pyproject.toml').read_text())['project']['dependencies']
    constraints = OUT/'constraints.txt'
    constraints.write_text('\n'.join(REQUIREMENTS)+'\n')
    subprocess.run([sys.executable,'-m','pip','install','-q','-c',str(constraints),*deps], check=True)
    for name, version in VERSIONS.items():
        if metadata.version(name) != version:
            raise RuntimeError(f'{name} cambiato: riavviare il runtime prima della raccolta.')
    entry = OUT/'collect_lowmem.py'
    entry.write_text(LOWMEM_DRIVER, encoding='utf8')
    env = dict(os.environ, PATCHALIASING_POPULATION='deliverable3', SITI_PER_BLOCCO='4',
               PYTHONPATH=str(pipe)+os.pathsep+os.environ.get('PYTHONPATH',''))
    device_args = [] if COLLECTION_DEVICE == 'auto' else ['--device', COLLECTION_DEVICE]

    def collect_step(args, logfile):
        with open(logfile, 'w', encoding='utf8') as handle:
            proc = subprocess.Popen([sys.executable,str(entry),*args], cwd=pipe, env=env,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end='', flush=True); handle.write(line); handle.flush()
            rc = proc.wait()
        if rc:
            raise RuntimeError(f'Raccolta fermata: exit {rc}. Log: {logfile}. Nessun fallback ad altri dati.')

    # A tiny independent run checks the cloned code, checkpoints and model access before spending
    # time on the complete population. It is never used as analysis input.
    run_id = uuid.uuid4().hex[:12]
    smoke_dir = OUT/f'collection_preflight_{run_id}'
    collect_step(['--out',str(smoke_dir),'--smoke','--models','p8-s8',*device_args,
                  '--population','deliverable3','--batch-size',str(COLLECTION_BATCH_SIZE)],
                 OUT/f'collection_preflight_{run_id}.log')
    print('Preflight raccolta SUPERATO.')

    # The complete run always uses a new destination. No old Parquet directory is inspected or
    # silently resumed, even if one happens to exist elsewhere in Drive.
    target = OUT/f'collection_fresh_{run_id}'/'data'
    for p, s in POPULATION:
        tag = f'p{p}-s{s}'
        collect_step(['--out',str(target),'--models',tag,*device_args,
                      '--batch-size',str(COLLECTION_BATCH_SIZE),'--population','deliverable3'],
                     OUT/f'collection_{tag}_{run_id}.log')
    collect_step(['--out',str(target),'--merge-only','--population','deliverable3'],
                 OUT/f'collection_merge_{run_id}.log')
    manifest_path = target/'collection_manifest.json'
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    manifest = json.loads(manifest_path.read_text(encoding='utf8'))
    if manifest.get('status') != 'complete' or set(manifest.get('planned_models', [])) != TAGS:
        raise RuntimeError('La raccolta fresca non ha superato il manifest completo; nessun dato parziale viene usato.')
    for name in ['contrasts','mdl_cells','collapse','sites']:
        if not (target/f'02_{name}.parquet').is_file():
            raise FileNotFoundError(target/f'02_{name}.parquet')
    DATA_DIR = str(target.resolve())
    COLLECTION_INFO = {'fresh':True,'run_id':run_id,'repository':REPO_URL,'revision':commit,
                       'data':DATA_DIR,'manifest_sha256':file_hash(manifest_path),
                       'population':sorted(TAGS)}
    atomic_json(COLLECTION_STATE, COLLECTION_INFO)
    atomic_json(OUT/'fresh_collection.json', COLLECTION_INFO)
    print('Raccolta fresca completa:', DATA_DIR)
else:
    if DATA_DIR:
        print('Uso la raccolta indicata dal test o da una chiamata esplicita:', DATA_DIR)
    else:
        if not COLLECTION_STATE.is_file():
            raise RuntimeError('Nessuna raccolta fresca registrata: eseguire una volta con COLLECT_FROM_SCRATCH=True.')
        COLLECTION_INFO = json.loads(COLLECTION_STATE.read_text(encoding='utf8'))
        DATA_DIR = str(Path(COLLECTION_INFO['data']).expanduser().resolve())
        manifest_path = Path(DATA_DIR)/'collection_manifest.json'
        required = [manifest_path, *(Path(DATA_DIR)/f'02_{name}.parquet' for name in ['contrasts','mdl_cells','collapse','sites'])]
        if not all(path.is_file() for path in required):
            raise RuntimeError('La raccolta fresca registrata è incompleta: usare COLLECT_FROM_SCRATCH=True per una nuova raccolta.')
        if file_hash(manifest_path) != COLLECTION_INFO.get('manifest_sha256'):
            raise RuntimeError('Manifest della raccolta fresca alterato: usare COLLECT_FROM_SCRATCH=True per una nuova raccolta.')
        manifest = json.loads(manifest_path.read_text(encoding='utf8'))
        if manifest.get('status') != 'complete' or set(manifest.get('planned_models', [])) != TAGS:
            raise RuntimeError('Manifest della raccolta fresca non completo: usare COLLECT_FROM_SCRATCH=True per una nuova raccolta.')
        for name in ['contrasts','mdl_cells','collapse','sites']:
            expected = manifest.get('merged', {}).get(name, {}).get('sha256')
            if not expected or file_hash(Path(DATA_DIR)/f'02_{name}.parquet') != expected:
                raise RuntimeError(f'Hash alterato per 02_{name}.parquet: usare COLLECT_FROM_SCRATCH=True per una nuova raccolta.')
        print('Uso la raccolta fresca registrata:', DATA_DIR)

## Dati e provenienza
La raccolta non cerca vecchi Parquet e non seleziona automaticamente una directory esistente.
La cella precedente crea una nuova raccolta completa con il codice e i checkpoint della revisione
dichiarata, poi il manifest viene verificato prima del caricamento. Ogni Parquet viene identificato
con SHA-256; le sorgenti originali e le raccolte precedenti restano immutate.

La risposta di A è
\(d=\log(R_k+0.01)-[\log(R_-+0.01)+\log(R_++0.01)]/2\).
L'inclusione non dipende da `live`, dall'effetto osservato o dalla sua direzione.
Il conteggio `legacy_live` rende visibile la differenza con le analisi precedenti.

In [ ]:
#@title 5. Caricamento, audit dei contrasti e della fase
FILENAMES = {name:f'02_{name}.parquet' for name in ['contrasts','mdl_cells','collapse','sites']}
if not DATA_DIR:
    raise RuntimeError('Nessuna raccolta fresca disponibile: rieseguire la cella 4 con COLLECT_FROM_SCRATCH=True.')
DATA = Path(DATA_DIR).expanduser().resolve()
SOURCE_HASHES = {name:file_hash(DATA/f) for name,f in FILENAMES.items()}
raw = {name:pd.read_parquet(DATA/f) for name,f in FILENAMES.items()}
CONTR, CONTR_AUDIT = canonical_contrasts(raw['contrasts'])
MDL = raw['mdl_cells'].query('model in @TAGS').copy().reset_index(drop=True)
COLL = raw['collapse'].query('model in @TAGS').copy().reset_index(drop=True)
SITES = raw['sites'].query('model in @TAGS').copy().reset_index(drop=True)
for name, frame in [('contrasts',CONTR),('mdl_cells',MDL),('collapse',COLL)]:
    if set(frame.model) != TAGS:
        raise ValueError(f'{name}: geometrie mancanti {sorted(TAGS-set(frame.model))}')
    pairs = frame[['model','P','S']].drop_duplicates()
    if len(pairs) != len(TAGS) or any(r.model != f'p{int(r.P)}-s{int(r.S)}' for r in pairs.itertuples()):
        raise ValueError(f'{name}: metadati P/S incoerenti')
if not np.allclose(CONTR.overlap, (CONTR.P-CONTR.S)/CONTR.P):
    raise ValueError('Overlap incoerente con P/S.')
if not np.isfinite(MDL.L_bits).all() or (MDL.L_bits <= 0).any() or not MDL.is_locked.isin([0,1]).all():
    raise ValueError('B richiede L_bits positivo e finito, is_locked binario.')
if not np.isfinite(COLL.z).all() or (COLL.z <= 0).any():
    raise ValueError('D1 richiede z positivo: stabilire una convenzione di misura, senza floor nascosti.')
if not np.isfinite(COLL.f).all():
    raise ValueError('Frequenze di collapse non finite.')
CONTR_AUDIT.update(background_groups=len(CONTR[['generator','bg_id']].drop_duplicates()),
                   legacy_background_groups=int(CONTR.bg_id.nunique()), phase_bins=int(CONTR.phase_bin.nunique()))
atomic_json(OUT/'data_manifest.json', {'code':CODE_SHA256,'source':str(DATA),'hashes':SOURCE_HASHES,
                                      'versions':VERSIONS,'contrasts':CONTR_AUDIT})
print(CONTR_AUDIT)
PHASE_AUDIT = CONTR[['f_lock','phase_idx','phase_angle','phase_bin']].drop_duplicates().sort_values(['f_lock','phase_idx'])
PHASE_AUDIT.to_csv(OUT/'phase_mapping.csv',index=False)
print('Mappatura fisica delle fasi (prime 24 righe; CSV completo):')
show(PHASE_AUDIT.head(24).assign(degrees=lambda x:np.rad2deg(x.phase_angle)))
GC = CONTR.groupby('model',as_index=False)[['P','S','overlap']].first().sort_values('model').reset_index(drop=True)
GC['xo'] = (GC.overlap-GC.overlap.mean())/.5
GC['xp'] = np.log(GC.P)-np.log(GC.P).mean()
show(GC)
del raw

## Modelli
**A:** \(d_i\sim t_4(\beta_{c[i]}+u_{k[i]}+u_{b[i]},\sigma)\),
\(\beta_c\sim N(\bar\beta+\delta_O\widetilde O_c+\delta_P\widetilde{\log P}_c,\tau)\).
Coefficienti \(t_4(0,0.5)\), scale half-\(t_4(0,0.5)\).
L'indice dello sfondo è la coppia generatore/realizzazione; il lock è la frequenza fisica.

**C:** stessa famiglia su \(y=-d\), senza covariate di configurazione, con
\(u_p\sim N(0,\sigma_\phi)\), \(\sigma_\phi\sim HalfNormal(0.25)\).
Le otto fette hanno lo stesso significato angolare a tutte le frequenze.

**B:** \(L_i\sim Gamma(k,k/\mu_i)\),
\(\log\mu_i=\alpha_0+\theta_{lock}I_i+u_{st[i]}+u_{geo[i]}\).
\(\alpha_0\sim N(\log\bar L,1)\), \(\theta_{lock}\sim N(0,0.5)\),
\(k\sim Gamma(shape=2,rate=0.1)\), scale di stadio/geometria half-normal(1).
Il prior dell'intercetta è data-informed, come nella tabella v0; il suo centro viene
fissato sul dataset di fit e mantenuto quando si producono i controlli predittivi.

**D1:** \(\log z_g(f)\sim N(\alpha_g+\theta_S I_S+\theta_P I_P,\sigma)\).
Coefficienti normal(0,1), residuo half-normal(1). Le etichette identificano i siti
esatti della griglia comune, tollerando soltanto l'arrotondamento del Parquet.
Restano i valori grezzi `z`, come nel testo; PPC per modalità e frequenza verificano
anche la variabilità che l'equazione originale non spiega.

**D2:** \(\hat f^F_{1,g}\sim N(\kappa_F\Delta^F_g,\sqrt{\sigma_F^2+1^2})\),
\(\kappa_F\sim N(0,1)\), \(\sigma_F\sim HalfNormal(5\,Hz)\).
Si conserva la Normal originale: il suo prior simmetrico ammette anche frequenze
negative. Il prior predictive lo espone, senza presentare quel supporto come fisico.


In [ ]:
#@title 6. Specifiche dei dati e costruttore dei modelli
@dataclass
class Spec:
    name: str
    family: str
    frame: pd.DataFrame
    arrays: dict
    dims: dict
    constants: dict
    strata: tuple
    identified: bool = True
    reason: str = ''

def subset_spec(spec, ix):
    # Dimensioni latenti e centro delle covariate restano quelli del disegno completo.
    return replace(spec, frame=spec.frame.iloc[ix].reset_index(drop=True),
                   arrays={k:np.asarray(v)[ix] for k,v in spec.arrays.items()})

def with_y(spec, y):
    return replace(spec, arrays={**spec.arrays,'y':np.asarray(y,float)})

def make_model(spec, factor=1.0, overlap=True, phase=True, use_s=True, use_p=True):
    a, n, c = spec.arrays, spec.dims, spec.constants
    with pm.Model() as model:
        def offset(name, sigma, size):
            if NON_CENTERED:
                return sigma*pm.Normal('z_'+name,0,1,shape=size)
            return pm.Normal('u_'+name,0,sigma,shape=size)
        if spec.family in ('A','C'):
            scale = .5*factor
            bbar = pm.StudentT('bbar',nu=4,mu=0,sigma=scale)
            tau = pm.HalfStudentT('tau',nu=4,sigma=scale)
            sk = pm.HalfStudentT('sigma_k',nu=4,sigma=scale)
            sb = pm.HalfStudentT('sigma_b',nu=4,sigma=scale)
            sd = pm.HalfStudentT('sigma',nu=4,sigma=scale)
            mean = bbar
            if spec.family == 'A':
                do = pm.StudentT('delta_O',nu=4,mu=0,sigma=scale) if overlap else 0.
                dp = pm.StudentT('delta_P',nu=4,mu=0,sigma=scale)
                mean = mean+do*np.asarray(c['xo'])+dp*np.asarray(c['xp'])
            beta = pm.Deterministic('beta_c',mean+offset('c',tau,n['c']))
            mu = beta[a['ci']]+offset('k',sk,n['k'])[a['ki']]+offset('b',sb,n['b'])[a['bi']]
            if spec.family == 'C' and phase:
                sphi = pm.HalfNormal('sigma_phi',sigma=.25*factor)
                up = pm.Deterministic('u_phase',offset('phase',sphi,N_PHASE))
                mu = mu+up[a['pi']]
            pm.StudentT('obs',nu=4,mu=mu,sigma=sd,observed=a['y'])
        elif spec.family == 'B':
            alpha = pm.Normal('alpha0',c['alpha_center'],1.)
            th = pm.Normal('theta_lock',0,.5*factor)
            k = pm.Gamma('k',alpha=2.,beta=.1)
            ss = pm.HalfNormal('sigma_st',1.*factor)
            sg = pm.HalfNormal('sigma_geo',1.*factor)
            mu = pm.math.exp(alpha+th*a['lock']+offset('st',ss,n['st'])[a['si']]
                             +offset('geo',sg,n['g'])[a['gi']])
            pm.Gamma('obs',alpha=k,beta=k/mu,observed=a['y'])
        elif spec.family == 'D1':
            alpha = pm.Normal('alpha_geo',0,1.*factor,shape=n['g'])
            ts = pm.Normal('theta_S',0,1.*factor) if use_s else 0.
            tp = pm.Normal('theta_P',0,1.*factor) if use_p else 0.
            sd = pm.HalfNormal('sigma',1.*factor)
            pm.Normal('obs',mu=alpha[a['gi']]+ts*a['s']+tp*a['p'],sigma=sd,observed=a['y'])
        elif spec.family == 'D2':
            kp = pm.Normal('kappa',0,1.*factor)
            sf = pm.HalfNormal('sigma_F',5.*factor)
            pm.Normal('obs',mu=kp*a['x'],sigma=pm.math.sqrt(sf**2+DELTA_F**2),observed=a['y'])
        else:
            raise ValueError(spec.family)
    return model

def core_specs(contr, mdl, coll):
    ci = pd.Categorical(contr.model,categories=GC.model).codes.astype('int32')
    ki,nk = codes(contr,['f_lock']); bi,nb = codes(contr,['generator','bg_id'])
    dims = dict(c=len(GC),k=nk,b=nb)
    const = dict(xo=GC.xo.tolist(),xp=GC.xp.tolist())
    arrays = dict(y=contr.d.to_numpy(float),ci=ci,ki=ki,bi=bi,pi=contr.phase_bin.to_numpy('int32'))
    sa = Spec('A','A',contr,arrays,dims,const,('f_lock','phase_bin','P','S','generator','model'))
    sc = replace(sa,name='C',family='C',arrays={**arrays,'y':-arrays['y']})
    si,nst = codes(mdl,['stage']); gi,ng = codes(mdl,['model'])
    sb = Spec('B','B',mdl,dict(y=mdl.L_bits.to_numpy(float),lock=mdl.is_locked.to_numpy(float),si=si,gi=gi),
              dict(st=nst,g=ng),dict(alpha_center=float(np.log(mdl.L_bits.mean()))),
              tuple(x for x in ['f_center','P','S','stage','is_locked','model','generator','phase_bin'] if x in mdl))
    coll = coll[coll.f.between(*BAND)].copy().reset_index(drop=True)
    gi,ng = codes(coll,['model'])
    f,p,s = coll.f.to_numpy(float),coll.P.to_numpy(float),coll.S.to_numpy(float)
    ks,kp = np.rint(f*s/FS),np.rint(f*p/FS)
    ons = ((ks>=1)&(abs(f-ks*FS/s)<=GRID_TOL)).astype(float)
    onp = ((kp>=1)&(abs(f-kp*FS/p)<=GRID_TOL)).astype(float)
    coll['grid_class'] = np.select([(ons==1)&(onp==0),(onp==1)&(ons==0),(ons==1)&(onp==1)],
                                   ['stride','patch','both'],default='neither')
    # Effetti di geometria + due etichette devono essere separabili nella likelihood.
    design = np.column_stack([np.eye(ng)[gi],ons,onp])
    identified = np.linalg.matrix_rank(design) == design.shape[1]
    sd = Spec('D1','D1',coll,dict(y=np.log(coll.z.to_numpy(float)),gi=gi,s=ons,p=onp),
              dict(g=ng),{},('f','P','S','mode','model','grid_class'),identified,
              '' if identified else 'etichette non separabili dagli effetti di geometria')
    return {'A':sa,'C':sc,'B':sb,'D1':sd}

SPECS = core_specs(CONTR,MDL,COLL)
if MODE == 'smoke':
    for name,spec in list(SPECS.items()):
        strata = ['model','generator'] if name in ['A','C'] else ['model','is_locked'] if name=='B' else ['model','grid_class']
        # A e C hanno esattamente le stesse righe e lo stesso ordine.
        ix = sample_balanced(spec.frame,SMOKE_ROWS,strata)
        SPECS[name] = subset_spec(spec,ix)
for name,spec in SPECS.items():
    print(name,len(spec.frame),'osservazioni',spec.dims)


## D2: fondamentale, armoniche mancanti e identificazione
Il file storico dei siti è usato per l'audit e, quando ci sono abbastanza geometrie,
per un fit **condizionale alla selezione teorica**. Si contano siti unici per geometria
sulle sole curve medie (`rep=-1`), unendo i due generatori e tenendo `pure` come controllo.
Repliche e curva media non vengono sommate come evidenze indipendenti.
Un primo sito vicino alla quinta armonica viene etichettato come tale, non diviso
silenziosamente per cinque per ottenere una pendenza vicina a uno.

Un input indipendente facoltativo `D2_MEASUREMENTS` è un CSV con una riga per
`model,branch`, colonne `P,S,f1,n_unique_sites,measurement_method,fundamental_verified`.
Il file deve avere un JSON omonimo con estensione `.provenance.json` contenente
`collapse_sha256`, `selection_uses_P_or_S: false`, `fundamental_method`,
`validation_report` (percorso a un rapporto esistente) e `validation_report_sha256`.
Il rapporto deve documentare recupero del fondamentale con armoniche mancanti e
prove nulle/controlli contro falsi pettini. Il notebook verifica schema, hash e
coerenza del disegno; **non certifica automaticamente il contenuto scientifico del rapporto**.
Una dichiarazione di provenienza non sostituisce tali prove.

Il minimo di dieci siti è conservato come salvaguardia aggiuntiva, non attribuito alla
tabella v0. Servono inoltre almeno tre geometrie e una serie che vari il parametro
di interesse tenendo fermo l'altro. Un risultato non identificato non equivale a una refutazione.


In [ ]:
#@title 7. Audit dei siti D2 e input indipendente facoltativo
def site_audit(sites):
    required = {'model','P','S','mode','rep','branch','sites','f1'}
    if not required <= set(sites):
        raise ValueError(f'Sites: mancano {sorted(required-set(sites))}')
    mean = sites[(sites.rep == -1)&sites['mode'].isin(['tsmixup','kernelsynth'])].copy()
    rows = []
    for (model,branch),g in mean.groupby(['model','branch'],sort=True):
        p,s = int(g.P.iloc[0]),int(g.S.iloc[0])
        vals = sorted({round(float(v),3) for value in g.sites.fillna('') for v in str(value).split()})
        delta = FS/(s if branch=='stride' else p)
        f1 = float(g.f1.median()) if g.f1.notna().any() else np.nan
        harmonic = int(round(f1/delta)) if np.isfinite(f1) else 0
        status = ('nessun sito' if not np.isfinite(f1) else
                  'armonica superiore / fondamentale assente' if harmonic>1 else
                  'candidato fondamentale, selezione teorica')
        rows.append(dict(model=model,branch=branch,P=p,S=s,f1=f1,x=delta,
                         n_unique_sites=len(vals),first_harmonic=harmonic,status=status))
    return pd.DataFrame(rows)

D2_AUDIT = site_audit(SITES)
D2_AUDIT.to_csv(OUT/'d2_site_audit.csv',index=False)
show(D2_AUDIT)
independent = None
if D2_MEASUREMENTS:
    path = Path(D2_MEASUREMENTS).expanduser().resolve()
    independent = pd.read_csv(path)
    required = {'model','branch','P','S','f1','n_unique_sites','measurement_method','fundamental_verified'}
    if not required <= set(independent):
        raise ValueError(f'Fondamentali indipendenti: mancano {sorted(required-set(independent))}')
    prov = json.loads(path.with_suffix('.provenance.json').read_text(encoding='utf8'))
    report = (path.parent/prov['validation_report']).resolve()
    if (prov.get('selection_uses_P_or_S') is not False or
        prov.get('collapse_sha256') != SOURCE_HASHES['collapse'] or
        not str(prov.get('fundamental_method','')).strip() or
        not report.is_file() or file_hash(report) != prov.get('validation_report_sha256')):
        raise ValueError('Provenienza D2 incompleta/incompatibile: servono misure indipendenti e rapporto verificabile.')
    if independent.duplicated(['model','branch']).any():
        raise ValueError('D2 indipendente: una sola misura per geometria e ramo.')
    if not independent.model.isin(TAGS).all() or not independent.branch.isin(['stride','patch']).all():
        raise ValueError('Popolazione/ramo D2 non riconosciuti.')
    for r in independent.itertuples():
        if r.model != f'p{int(r.P)}-s{int(r.S)}':
            raise ValueError('Geometria D2 incoerente con il tag.')
    if not independent.fundamental_verified.isin([True,False,0,1]).all():
        raise ValueError('fundamental_verified deve essere booleano.')
    if independent.measurement_method.isna().any() or independent.measurement_method.str.strip().eq('').any():
        raise ValueError('Metodo di misura assente.')
    vals = independent[['f1','n_unique_sites']].to_numpy(float)
    if (not np.isfinite(vals).all() or (vals<=0).any() or
        not np.equal(vals[:,1],np.floor(vals[:,1])).all()):
        raise ValueError('Fondamentali e conteggi indipendenti devono essere validi e positivi.')
    SOURCE_HASHES.update(d2_measurements=file_hash(path),d2_provenance=file_hash(path.with_suffix('.provenance.json')),
                         d2_validation_report=file_hash(report))

D2_STATUS = {}
for branch in ['stride','patch']:
    name = 'D2_'+branch
    if independent is None:
        g = D2_AUDIT[(D2_AUDIT.branch==branch)&D2_AUDIT.f1.notna()].copy()
        # Le armoniche superiori non diventano falsi fondamentali.
        g = g[g.first_harmonic==1].reset_index(drop=True)
        measured = False
        reason = 'fondamentale non verificato indipendentemente; siti selezionati sulla teoria'
    else:
        g = independent[(independent.branch==branch)&independent.fundamental_verified.astype(bool)].copy()
        g['x'] = FS/(g.S if branch=='stride' else g.P)
        measured,reason = True,''
    varied,fixed = ('S','P') if branch=='stride' else ('P','S')
    series = bool(len(g) and g.groupby(fixed)[varied].nunique().max() >= 3)
    counts = int(g.n_unique_sites.sum()) if len(g) else 0
    enough = len(g)>=3 and counts>=MIN_SITES and series
    if not enough:
        reason += f'; disegno insufficiente: {len(g)} geometrie, {counts} siti unici, serie controllata={series}'
    D2_STATUS[name] = dict(identified=bool(measured and enough),reason=reason.strip('; '),n_unique_sites=counts)
    if len(g)>=3:
        SPECS[name] = Spec(name,'D2',g.reset_index(drop=True),
                          dict(y=g.f1.to_numpy(float),x=g.x.to_numpy(float)),{}, {},
                          ('model','P','S'),bool(measured and enough),reason.strip('; '))
print('Identificazione D2:',D2_STATUS)


In [ ]:
#@title 8. Campionamento, diagnostiche complete e checkpoint atomici
if ENGINE == 'auto':
    try:
        import numpyro, jax
        ENGINE_USED = 'numpyro'
    except ImportError:
        ENGINE_USED = 'pymc'
else:
    ENGINE_USED = ENGINE
if ENGINE_USED == 'numpyro':
    import jax, numpyro
    jax.config.update('jax_enable_x64',True)
    VERSIONS.update(jax=jax.__version__,numpyro=numpyro.__version__)
    print('Dispositivi JAX:',jax.devices())

def spec_digest(spec):
    digest = hashlib.sha256()
    digest.update(json_hash({'family':spec.family,'dims':spec.dims,'constants':spec.constants,
                             'strata':spec.strata,'identified':spec.identified,'reason':spec.reason}).encode())
    digest.update(json_hash([(str(k),str(v)) for k,v in spec.frame.dtypes.items()]).encode())
    digest.update(pd.util.hash_pandas_object(spec.frame,index=True).to_numpy('uint64').tobytes())
    for key,value in sorted(spec.arrays.items()):
        value = np.ascontiguousarray(value)
        digest.update(json_hash([key,str(value.dtype),value.shape]).encode()); digest.update(value.tobytes())
    return digest.hexdigest()

def fit_identity(spec,options,draws,tune,chains,seed):
    # Hash dei code object ATTUALMENTE caricati: cambia anche dopo una modifica in Colab.
    # Esclude filename/numero di cella, così un riavvio non distrugge il riuso legittimo.
    import types
    def constant_record(value):
        if isinstance(value,types.CodeType):
            return code_record(value)
        if isinstance(value,(tuple,list)):
            return [constant_record(v) for v in value]
        if isinstance(value,(set,frozenset)):
            return sorted((constant_record(v) for v in value),key=lambda v:json.dumps(v,sort_keys=True))
        if isinstance(value,dict):
            return {str(k):constant_record(v) for k,v in sorted(value.items())}
        return repr(value)
    def code_record(code):
        return dict(bytecode=code.co_code.hex(),names=code.co_names,variables=code.co_varnames,
                    free=code.co_freevars,cell=code.co_cellvars,argcount=code.co_argcount,
                    kwonly=code.co_kwonlyargcount,flags=code.co_flags,
                    constants=[constant_record(v) for v in code.co_consts])
    current_code=json_hash({f.__name__:dict(body=code_record(f.__code__),defaults=constant_record(f.__defaults__),
                                          kwdefaults=constant_record(f.__kwdefaults__)) for f in
                           [make_model,subset_spec,canonical_contrasts,core_specs,site_audit,sample_fit]})
    return dict(code=CODE_SHA256,runtime_code=current_code,data=spec_digest(spec),sources=SOURCE_HASHES,versions=VERSIONS,
                constants=dict(fs=FS,band=BAND,epsilon=EPS,n_phase=N_PHASE,delta_f=DELTA_F,grid_tol=GRID_TOL),
                family=spec.family,options=options,draws=draws,tune=tune,chains=chains,
                block=BLOCK_CHAINS,seed=seed,engine=ENGINE_USED,non_centered=NON_CENTERED,
                dense_mass=DENSE_MASS,target_accept=.95,max_treedepth=12,mode=MODE)

def diagnostics(idata):
    # Iterare le variabili evita il broadcasting cartesiano di Dataset.to_array().
    result = dict(rhat=np.nan,ess_bulk=np.nan,ess_tail=np.nan,divergences=None,ok=False)
    try:
        names = list(idata.posterior.data_vars)
        metrics = [az.rhat(idata,var_names=names), az.ess(idata,var_names=names,method='bulk'),
                   az.ess(idata,var_names=names,method='tail')]
        flat = [np.concatenate([np.asarray(v).ravel() for v in ds.data_vars.values()]) for ds in metrics]
        if not all(len(x) and np.isfinite(x).all() for x in flat):
            result['reason']='diagnostiche non finite'; return result
        if 'sample_stats' not in idata.groups() or 'diverging' not in idata.sample_stats:
            result['reason']='diagnostica divergenze mancante'; return result
        result.update(rhat=float(flat[0].max()),ess_bulk=float(flat[1].min()),ess_tail=float(flat[2].min()),
                      divergences=int(idata.sample_stats.diverging.sum()))
        result['ok'] = bool(idata.posterior.sizes['chain']>=4 and result['rhat']<RHAT_MAX and
                            min(result['ess_bulk'],result['ess_tail'])>ESS_MIN and result['divergences']==0)
    except (ValueError,TypeError,KeyError) as exc:
        result['reason']=str(exc)
    return result

def write_block(idata,path,identity,start):
    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_name(path.stem+'.'+uuid.uuid4().hex+'.tmp.nc')
    idata.to_netcdf(temp,engine='h5netcdf')
    sha = file_hash(temp)
    os.replace(temp,path)
    atomic_json(path.with_suffix('.json'),dict(identity=identity,start=start,sha256=sha,
        chains=idata.posterior.sizes['chain'],draws=idata.posterior.sizes['draw']))

def read_block(path,identity,start,chains,draws):
    path = Path(path); meta = path.with_suffix('.json')
    if not path.is_file() or not meta.is_file():
        return None
    try:
        saved=json.loads(meta.read_text())
        if (saved.get('identity')!=identity or saved.get('start')!=start or
            saved.get('chains')!=chains or saved.get('draws')!=draws or saved.get('sha256')!=file_hash(path)):
            return None
        loaded=az.from_netcdf(path,engine='h5netcdf')
        for group in loaded.groups():
            getattr(loaded,group).load()
        loaded.close()
        if loaded.posterior.sizes['chain']!=chains or loaded.posterior.sizes['draw']!=draws:
            return None
        return loaded
    except (OSError,ValueError,KeyError):
        return None

FIT_LOG=[]
def sample_fit(spec,label,options=None,draws=None,tune=None,seed=SEED):
    if MODE=='preflight':
        raise RuntimeError('MCMC disabilitato in preflight.')
    options=dict(options or {})
    draws=DRAWS if draws is None else draws; tune=TUNE if tune is None else tune
    ident=fit_identity(spec,options,draws,tune,CHAINS,seed)
    identity=json_hash(ident)
    directory=OUT/'ckpt'/identity
    directory.mkdir(parents=True,exist_ok=True)
    atomic_json(directory/'fit.json',ident)
    blocks=[]
    for start in range(0,CHAINS,BLOCK_CHAINS):
        count=min(BLOCK_CHAINS,CHAINS-start)
        path=directory/f'chains_{start:02d}.nc'
        block=read_block(path,identity,start,count,draws)
        if block is None:
            print(f'{label}: catene {start+1}-{start+count}, {draws}+{tune} iterazioni, n={len(spec.frame)}',flush=True)
            kwargs=dict(draws=draws,tune=tune,chains=count,random_seed=seed+start,
                        target_accept=.95,progressbar=True,idata_kwargs={'log_likelihood':False})
            if ENGINE_USED=='numpyro':
                kwargs.update(nuts_sampler='numpyro',nuts_sampler_kwargs={
                    'chain_method':'vectorized','nuts_kwargs':{'max_tree_depth':12,'dense_mass':DENSE_MASS}})
            else:
                kwargs.update(cores=min(count,os.cpu_count() or 1),nuts={'max_treedepth':12},
                              init='jitter+adapt_full' if DENSE_MASS else 'jitter+adapt_diag')
            with make_model(spec,**options):
                block=pm.sample(**kwargs)
            # Un errore di salvataggio resta visibile, non viene dichiarato un checkpoint inesistente.
            write_block(block,path,identity,start)
        else:
            print(f'{label}: checkpoint verificato, catene {start+1}-{start+count}')
        blocks.append(block)
    result=blocks[0] if len(blocks)==1 else az.concat(*blocks,dim='chain',reset_dim=True)
    FIT_LOG.append(dict(label=label,identity=identity,**diagnostics(result)))
    pd.DataFrame(FIT_LOG).to_csv(OUT/'sampling_diagnostics.csv',index=False)
    return result

def thin_posterior(idata,limit):
    post=idata.posterior.stack(sample=('chain','draw'))
    indices=np.linspace(0,post.sizes['sample']-1,min(limit,post.sizes['sample']),dtype=int)
    post=post.isel(sample=indices).reset_index('sample',drop=True).rename({'sample':'draw'})
    return post.assign_coords(draw=np.arange(len(indices))).expand_dims(chain=[0]).transpose('chain','draw',...)

print('Motore:',ENGINE_USED,'; NCP:',NON_CENTERED,'; metrica densa:',DENSE_MASS)
print('Fit principali:',{k:len(v.frame) for k,v in SPECS.items()})
print('Full include recovery in due scenari per famiglia, tre repliche, LOO e tre scale per modello.')


## Controlli predittivi a priori
Le probabilità della tabella sono ricalcolate analiticamente. La simulazione seguente
usa **il costruttore reale di ogni modello**, comprese le covariate e gli effetti di gruppo.
Si lavora su un sottoinsieme dichiarato di osservazioni per limitare la memoria, mantenendo
le dimensioni latenti e il centro delle covariate del disegno.

Il gate automatico verifica simulazioni finite e copertura grossolana del supporto osservato
(almeno 95% delle misure nell'intervallo predittivo marginale 0,1%-99,9%). È uno screening,
non una dimostrazione che il prior sia appropriato: quantili, code e frequenze negative di D2
restano disponibili per l'ispezione. Questi controlli precedono i fit; con archivi esistenti
non possono essere presentati retroattivamente come eseguiti prima della raccolta.


In [ ]:
#@title 9. Priori: probabilità e simulazioni complete dei modelli
PRIOR_PROBS=pd.DataFrame([
    ('H1 supportata',stats.t.cdf(LOG08,df=4,scale=.5),.34),
    ('H1 rifiutata',stats.t.cdf(LOG11,df=4,scale=.5)-stats.t.cdf(-LOG11,df=4,scale=.5),.14),
    ('H2 supportata',stats.halfnorm.cdf(LOG11,scale=.25),.30),
    ('H3a/H3b',stats.norm.cdf(1.1)-stats.norm.cdf(.9),.05),
    ('M1 (segno corretto)',.5,.50),
    ('B: regola aggiunta',stats.norm.sf(LOG12,scale=.5),np.nan)],columns=['regola','probabilita','tabella_v0'])
show(PRIOR_PROBS)
assert ((PRIOR_PROBS.probabilita-PRIOR_PROBS.tabella_v0).dropna().abs()<.01).all()
PRIOR_ROWS=[]; PRIOR_OK={}; PRIOR_SAMPLES={}
if MODE!='preflight':
    for name,spec in SPECS.items():
        strata=['model']
        ix=sample_balanced(spec.frame,min(256,len(spec.frame)),strata)
        mini=subset_spec(spec,ix)
        with make_model(mini):
            prior=pm.sample_prior_predictive(draws=PRIOR_DRAWS,var_names=['obs'],random_seed=SEED)
        sim=prior.prior_predictive.obs.values.ravel()
        finite=bool(np.isfinite(sim).all())
        lo,hi=np.quantile(sim,[.001,.999]) if finite else (np.nan,np.nan)
        coverage=float(np.mean((mini.arrays['y']>=lo)&(mini.arrays['y']<=hi)))
        PRIOR_OK[name]=bool(finite and coverage>=.95)
        PRIOR_ROWS.append(dict(model=name,rows=len(ix),draws=PRIOR_DRAWS,finite=finite,
            q001=lo,q999=hi,observed_coverage=coverage,negative_fraction=float(np.mean(sim<0)),ok=PRIOR_OK[name]))
        PRIOR_SAMPLES[name]=sim[::max(1,len(sim)//5000)]
        del prior; gc.collect()
    PRIOR_TABLE=pd.DataFrame(PRIOR_ROWS)
    show(PRIOR_TABLE); PRIOR_TABLE.to_csv(OUT/'prior_predictive.csv',index=False)
else:
    print('PREFLIGHT: simulazioni predittive e fit non avviati.')


In [ ]:
#@title 10. Fit principali
FITS={}; MAIN_DIAG={}
if MODE!='preflight':
    for name,spec in SPECS.items():
        FITS[name]=sample_fit(spec,'main_'+name)
        MAIN_DIAG[name]=diagnostics(FITS[name])
    show(pd.DataFrame([dict(model=n,**d) for n,d in MAIN_DIAG.items()]))


## Controlli predittivi a posteriori
I draw vengono selezionati **prima** della simulazione e generati in blocchi piccoli.
Non viene materializzato l'array completo catene × draw × osservazioni. Si verificano
media e dispersione per ogni livello disponibile di frequenza, fase fisica, P, S e generatore,
più gli strati propri del modello. D2 è incluso. Un livello con una sola osservazione
ha un controllo della misura, ma non una stima di dispersione.

Per B la raccolta MDL aggrega già sfondi e fasi: fase e generatore non sono ricostruibili
da quei Parquet. Il CSV `ppc_coverage` dichiara questo limite; non si inventano etichette.
Lo screening richiede copertura al 95% degli intervalli per almeno il 95% dei livelli
di **ciascuno** strato/statistica, più i quantili globali. I livelli non vengono troncati ai primi 40.


In [ ]:
#@title 11. PPC a memoria limitata, per ogni modello e strato
def ppc_check(spec,idata):
    posterior=thin_posterior(idata,PPC_DRAWS)
    plans=[]; observed={}; predictions={}
    y=spec.arrays['y']
    for col in spec.strata:
        if col not in spec.frame or spec.frame[col].isna().any():
            raise ValueError(f'PPC {spec.name}: strato {col} mancante/incompleto')
        group,levels=pd.factorize(spec.frame[col],sort=True)
        counts=np.bincount(group,minlength=len(levels))
        mean=np.bincount(group,weights=y)/counts
        sd=np.sqrt(np.maximum(0,np.bincount(group,weights=y*y)/counts-mean*mean))
        for k,level in enumerate(levels):
            for metric,values in [('mean',mean),('sd',sd)]:
                if metric=='sd' and counts[k]<2:
                    continue
                key=(col,str(level),metric)
                observed[key]=float(values[k]); predictions[key]=[]
        plans.append((col,group,levels,counts))
    for metric,q in [('q05',.05),('q50',.5),('q95',.95)]:
        key=('global','all',metric); observed[key]=float(np.quantile(y,q)); predictions[key]=[]
    for start in range(0,posterior.sizes['draw'],PPC_BATCH_DRAWS):
        batch=posterior.isel(draw=slice(start,start+PPC_BATCH_DRAWS))
        with make_model(spec):
            pp=pm.sample_posterior_predictive(batch,var_names=['obs'],random_seed=SEED+start,progressbar=False)
        rep=pp.posterior_predictive.obs.values.reshape(-1,len(y))
        if not np.isfinite(rep).all():
            raise ValueError(f'PPC {spec.name}: simulazioni non finite')
        for col,group,levels,counts in plans:
            means=np.stack([np.bincount(group,weights=row)/counts for row in rep])
            squares=np.stack([np.bincount(group,weights=row*row)/counts for row in rep])
            sds=np.sqrt(np.maximum(0,squares-means*means))
            for k,level in enumerate(levels):
                for metric,values in [('mean',means),('sd',sds)]:
                    key=(col,str(level),metric)
                    if key in predictions:
                        predictions[key].extend(values[:,k].tolist())
        for metric,q in [('q05',.05),('q50',.5),('q95',.95)]:
            predictions[('global','all',metric)].extend(np.quantile(rep,q,axis=1).tolist())
        del pp,rep; gc.collect()
    rows=[]
    for key,value in observed.items():
        lo,hi=np.quantile(predictions[key],[.025,.975])
        rows.append(dict(model=spec.name,stratum=key[0],level=key[1],metric=key[2],observed=value,
                         low=float(lo),high=float(hi),ok=bool(lo<=value<=hi)))
    return pd.DataFrame(rows)

PPC_TABLE=pd.DataFrame(); PPC_OK={}; PPC_COVERAGE=[]
if MODE!='preflight':
    tables=[]
    for name,spec in SPECS.items():
        table=ppc_check(spec,FITS[name]); tables.append(table)
        rates=table.groupby(['stratum','metric']).ok.mean()
        PPC_OK[name]=bool(len(rates) and (rates>=.95).all())
        for st,metric in rates.index:
            PPC_COVERAGE.append(dict(model=name,stratum=st,metric=metric,coverage=float(rates.loc[(st,metric)])))
    PPC_TABLE=pd.concat(tables,ignore_index=True)
    PPC_TABLE.to_csv(OUT/'ppc_details.csv',index=False)
    pd.DataFrame(PPC_COVERAGE).to_csv(OUT/'ppc_coverage.csv',index=False)
    show(pd.DataFrame([dict(model=n,ok=v) for n,v in PPC_OK.items()]))
    print('B: fase e generatore non disponibili nella tabella MDL aggregata.')


## Recupero di parametri
Ogni famiglia ha due scenari, vicino al nullo e con un effetto sostanziale. In particolare
**C simula e rifitta sigma_phi**, e D2 verifica sia kappa=1 sia una pendenza diversa.
Il fit usa il disegno osservato (un campione stratificato limitato per A/B/C/D1),
non soltanto una likelihood con nomi simili. Per un ramo D2 assente non si inventa
un disegno empirico: non può ricevere un verdetto.

Nel full si ripetono tre simulazioni per scenario. Lo screening richiede convergenza
e copertura del vero nell'intervallo al 95% per almeno l'80% delle repliche di ogni
parametro/scenario. Si riporta anche la larghezza: copertura con intervalli molto larghi
non dimostra potenza. Le regole finali restano basate sulle probabilità posteriori,
non sul semplice successo del recovery.


In [ ]:
#@title 12. Recovery specifico per A, B, C, D1 e ciascun ramo D2
def simulate_known(spec,effect,seed):
    rng=np.random.default_rng(seed); a,n,c=spec.arrays,spec.dims,spec.constants
    if spec.family in ('A','C'):
        bbar=-.3 if effect else 0.
        beta=bbar+.12*rng.normal(size=n['c'])
        true={'bbar':bbar}
        if spec.family=='A':
            do,dp=(.3,-.15) if effect else (0.,0.)
            beta=beta+do*np.asarray(c['xo'])+dp*np.asarray(c['xp'])
            true.update(delta_O=do,delta_P=dp)
        mu=beta[a['ci']]+.12*rng.normal(size=n['k'])[a['ki']]+.1*rng.normal(size=n['b'])[a['bi']]
        if spec.family=='C':
            sphi=.25 if effect else .02
            mu=mu+sphi*rng.normal(size=N_PHASE)[a['pi']]
            true={'sigma_phi':sphi}
        y=mu+.4*rng.standard_t(4,len(mu))
    elif spec.family=='B':
        th=.35 if effect else 0.
        mu=np.exp(c['alpha_center']+th*a['lock']+.15*rng.normal(size=n['st'])[a['si']]
                  +.15*rng.normal(size=n['g'])[a['gi']])
        y=rng.gamma(shape=20.,scale=mu/20.); true={'theta_lock':th}
    elif spec.family=='D1':
        ts,tp=(-.7,-.5) if effect else (0.,0.)
        mu=.3*rng.normal(size=n['g'])[a['gi']]+ts*a['s']+tp*a['p']
        y=mu+.4*rng.normal(size=len(mu)); true={'theta_S':ts,'theta_P':tp}
    else:
        kp=1. if effect else .65
        y=kp*a['x']+np.sqrt(2.**2+DELTA_F**2)*rng.normal(size=len(a['x']))
        true={'kappa':kp}
    return with_y(spec,y),true

RECOVERY_TABLE=pd.DataFrame(); RECOVERY_OK={}
if MODE!='preflight':
    rows=[]
    for name,spec in SPECS.items():
        strata=['model','phase_bin'] if spec.family=='C' else ['model']
        ix=sample_balanced(spec.frame,min(3000 if MODE=='full' else SMOKE_ROWS,len(spec.frame)),strata,seed=SEED+7)
        design=subset_spec(spec,ix)
        for effect in [False,True]:
            for repeat in range(RECOVERY_REPEATS):
                seed=SEED+1000+100*repeat+int(effect)
                sim,true=simulate_known(design,effect,seed)
                idata=sample_fit(sim,f'recovery_{name}_{int(effect)}_{repeat}',draws=VALID_DRAWS,tune=VALID_DRAWS,seed=seed)
                diag=diagnostics(idata)
                for parameter,value in true.items():
                    draws=idata.posterior[parameter].values.ravel(); lo,hi=np.quantile(draws,[.025,.975])
                    rows.append(dict(model=name,scenario='effect' if effect else 'near_null',repeat=repeat,
                        parameter=parameter,true=value,median=float(np.median(draws)),low=float(lo),high=float(hi),
                        width=float(hi-lo),covered=bool(lo<=value<=hi),converged=diag['ok']))
                del idata; gc.collect()
    RECOVERY_TABLE=pd.DataFrame(rows)
    RECOVERY_TABLE.to_csv(OUT/'parameter_recovery.csv',index=False)
    for name,g in RECOVERY_TABLE.groupby('model'):
        rates=g.groupby(['scenario','parameter']).covered.mean()
        RECOVERY_OK[name]=bool(set(g.scenario)=={'effect','near_null'} and g.converged.all() and (rates>=.8).all())
    show(RECOVERY_TABLE)


## Confronti LOO
I confronti usano lo stesso sottoinsieme stratificato e lo stesso ordine delle osservazioni
per tutte le varianti. Nel full il limite è 20.000 righe: questo è un confronto predittivo
sul sottoinsieme dichiarato, non l'ELPD dell'intero archivio. I fit principali usano tutti i dati.
Ogni fit LOO deve superare le stesse diagnostiche. PSIS viene calcolato per blocchi di
osservazioni per evitare un'unica matrice log-likelihood enorme.

Per confrontare i modelli si usa
\(dSE=\sqrt{N\operatorname{Var}(\ell_{1,i}-\ell_{0,i})}\), con coppie sullo stesso punto.
La vittoria operativa richiede `delta_elpd > 2*dSE`; la soglia di Pareto-k è quella
restituita da ArviZ per il numero di draw, al massimo 0,7. Un pareggio è inconclusivo,
un confronto senza diagnostiche valide è non verificato. Il LOO di C viene riportato
separatamente dalla regola ROPE di H2.

Riferimento: [ArviZ 0.22, confronto delle differenze ELPD](https://python.arviz.org/en/v0.22.0/_modules/arviz/stats/stats.html).
Il LOO è per osservazione condizionata ai gruppi del disegno; non misura la generalizzazione
a una nuova geometria mai osservata.


In [ ]:
#@title 13. LOO a blocchi, con diagnostiche e differenze appaiate
def pointwise_loo(spec,idata,options,block_rows=256):
    parts=[]; ks=[]; p_loo=0.; threshold=.7
    for start in range(0,len(spec.frame),block_rows):
        mini=subset_spec(spec,np.arange(start,min(start+block_rows,len(spec.frame))))
        with make_model(mini,**options):
            ll=pm.compute_log_likelihood(idata,extend_inferencedata=False,progressbar=False)
        scored=az.loo(az.InferenceData(posterior=idata.posterior,log_likelihood=ll),pointwise=True)
        parts.extend(np.asarray(scored.loo_i).ravel().tolist())
        ks.extend(np.asarray(scored.pareto_k).ravel().tolist())
        p_loo+=float(scored.p_loo)
        threshold=min(threshold,float(scored.good_k))
        del ll,scored; gc.collect()
    point=np.asarray(parts); k=np.asarray(ks)
    valid=bool(np.isfinite(point).all() and np.isfinite(k).all() and (k<threshold).all())
    return dict(point=point,k=k,elpd=float(point.sum()),se=float(np.sqrt(len(point)*np.var(point))),
                p_loo=p_loo,psis_ok=valid,k_limit=threshold,k_max=float(np.max(k)))

def paired_loo_difference(first,second):
    a,b=np.asarray(first,float),np.asarray(second,float)
    if a.shape!=b.shape or a.ndim!=1 or len(a)<2 or not np.isfinite(a-b).all():
        raise ValueError('LOO: contributi mancanti, non allineati o non finiti.')
    difference=a-b
    return float(difference.sum()),float(np.sqrt(len(difference)*np.var(difference)))

LOO_RESULTS={}; LOO_ROWS_TABLE=[]; LOO_VALID={}; LOO_WINS={}
if MODE!='preflight':
    groups=[('M1','A',{'with':{},'without':{'overlap':False}},'with'),
            ('H2','C',{'with':{},'without':{'phase':False}},'with'),
            ('H3','D1',{'both':{},'stride':{'use_p':False},'patch':{'use_s':False},
                         'neither':{'use_s':False,'use_p':False}},'both')]
    for claim,name,variants,reference in groups:
        spec=SPECS[name]
        strata=['model','generator','phase_bin'] if name in ['A','C'] else ['model','mode','grid_class']
        # Nei piccoli smoke non tutti gli incroci sono popolati; il limite si alza solo quanto basta.
        groups_n=spec.frame.groupby(strata,observed=True).ngroups
        ix=sample_balanced(spec.frame,min(len(spec.frame),max(LOO_ROWS,groups_n)),strata,seed=SEED+11)
        design=subset_spec(spec,ix)
        pd.DataFrame({'source_row':ix}).to_csv(OUT/f'loo_{claim}_rows.csv',index=False)
        scores={}; valid=True
        for variant,options in variants.items():
            idata=sample_fit(design,f'loo_{claim}_{variant}',options,VALID_DRAWS,VALID_DRAWS,seed=SEED+11)
            diag=diagnostics(idata)
            score=pointwise_loo(design,idata,options)
            score['converged']=diag['ok']
            valid=bool(valid and diag['ok'] and score['psis_ok'])
            scores[variant]=score
            np.savez_compressed(OUT/f'loo_{claim}_{variant}.npz',loo_i=score['point'],pareto_k=score['k'],row=ix)
            del idata; gc.collect()
        wins=[]
        for alternative in variants:
            if alternative==reference:
                continue
            diff,dse=paired_loo_difference(scores[reference]['point'],scores[alternative]['point'])
            winner=bool(valid and diff>2*dse)
            wins.append(winner)
            LOO_ROWS_TABLE.append(dict(claim=claim,reference=reference,alternative=alternative,
                n=len(ix),delta_elpd=diff,dse=dse,valid=valid,wins=winner,
                k_max=max(scores[reference]['k_max'],scores[alternative]['k_max'])))
        LOO_VALID[claim]=valid; LOO_WINS[claim]=bool(valid and all(wins))
        LOO_RESULTS[claim]=scores
    LOO_TABLE=pd.DataFrame(LOO_ROWS_TABLE)
    show(LOO_TABLE); LOO_TABLE.to_csv(OUT/'loo_comparisons.csv',index=False)


In [ ]:
#@title 14. Probabilità delle regole, senza promuoverle ancora a verdetti
def probabilities(spec,idata):
    def values(name):
        return idata.posterior[name].values.ravel()
    def row(claim,param,draws,support,refute,transform=False):
        estimate=np.exp(draws) if transform else draws
        lo,hi=np.quantile(estimate,[.025,.975])
        return dict(claim=claim,model=spec.name,parameter=param,median=float(np.median(estimate)),
                    low=float(lo),high=float(hi),p_support=float(np.mean(support)),p_refute=float(np.mean(refute)))
    if spec.family=='A':
        b,d=values('bbar'),values('delta_O')
        return [row('H1_beh','exp(bbar)',b,b<LOG08,abs(b)<LOG11,True),
                row('M1','delta_O',d,d>0,d<0)]
    if spec.family=='B':
        d=values('theta_lock')
        return [row('H1_repr','exp(theta_lock)',d,d>LOG12,abs(d)<LOG11,True)]
    if spec.family=='C':
        d=values('sigma_phi')
        return [row('H2','sigma_phi',d,d<LOG11,d>=LOG11)]
    if spec.family=='D1':
        s,p=values('theta_S'),values('theta_P')
        # Evento congiunto, non due probabilità marginali entrambe >= .95.
        return [dict(claim='H3_location',model=spec.name,parameter='theta_S < 0 AND theta_P < 0',
                     median=np.nan,low=np.nan,high=np.nan,p_support=float(np.mean((s<0)&(p<0))),
                     p_refute=float(max(np.mean(s>=0),np.mean(p>=0))))]
    d=values('kappa')
    return [row('H3a' if spec.name.endswith('stride') else 'H3b','kappa',d,abs(d-1)<.1,abs(d-1)>=.1)]

def probability_class(support,refute):
    if not np.isfinite(support) or not np.isfinite(refute):
        return 'NON VERIFICATA'
    if support>=PROB:
        return 'SUPPORTATA'
    if refute>=PROB:
        return 'RIFIUTATA'
    return 'INCONCLUSIVA'

MAIN_PROBS=[]; D1_EFFECTS=[]
if MODE!='preflight':
    MAIN_PROBS=[r for name,spec in SPECS.items() for r in probabilities(spec,FITS[name])]
    show(pd.DataFrame(MAIN_PROBS))
    for parameter in ['theta_S','theta_P']:
        draws=FITS['D1'].posterior[parameter].values.ravel()
        lo,hi=np.quantile(draws,[.025,.975])
        D1_EFFECTS.append(dict(parameter=parameter,median=float(np.median(draws)),low=float(lo),high=float(hi),
                               p_negative=float(np.mean(draws<0))))
    show(pd.DataFrame(D1_EFFECTS))
    pd.DataFrame(D1_EFFECTS).to_csv(OUT/'d1_coefficients.csv',index=False)


## Sensibilità ai priori
Per **ogni** modello si usano gli stessi dati del fit principale a fattori 0,5, 1 e 2
rispetto alle scale originali. Per A e l'effetto di B questo dà 0,25 / 0,5 / 1;
per sigma_phi 0,125 / 0,25 / 0,5; per i coefficienti D1/D2 0,5 / 1 / 2.
Le scale nuisance vengono variate con lo stesso fattore; i centri e la shape Gamma
rimangono invariati. Il baseline è il fit principale, non un altro sottoinsieme.

Tutte e tre le varianti devono convergere. Lo screening richiede escursione delle
probabilità <=0,10 e la stessa classe decisionale in tutte le varianti; due varianti
convergenti non bastano a cancellarne una terza fallita. Le vittorie LOO sono riportate
al prior originale: questa analisi controlla le probabilità delle regole, non rivaluta
tutte le classifiche LOO a ciascun prior.


In [ ]:
#@title 15. Sensibilità su tutti i modelli, senza gate fissati a True
SENS_ROWS=[]; SENS_OK={}
if MODE!='preflight':
    for name,spec in SPECS.items():
        for factor in [.5,1.,2.]:
            idata=FITS[name] if factor==1 else sample_fit(spec,f'sensitivity_{name}_{factor}',{'factor':factor})
            diag=diagnostics(idata)
            for row in probabilities(spec,idata):
                SENS_ROWS.append(dict(**row,factor=factor,converged=diag['ok'],
                                      decision=probability_class(row['p_support'],row['p_refute'])))
            if factor!=1:
                del idata; gc.collect()
    SENS_TABLE=pd.DataFrame(SENS_ROWS)
    for claim,g in SENS_TABLE.groupby('claim'):
        SENS_OK[claim]=bool(set(g.factor)=={.5,1.,2.} and g.converged.all() and
            g.p_support.max()-g.p_support.min()<=.10 and g.p_refute.max()-g.p_refute.min()<=.10 and
            g.decision.nunique()==1)
    show(SENS_TABLE); SENS_TABLE.to_csv(OUT/'prior_sensitivity.csv',index=False)


In [ ]:
#@title 16. Verdetti: una sola funzione per tutti i modelli
def final_decision(row,gates,identified=True,mode=MODE,loo_wins=None):
    if mode!='full':
        return 'NON RIPORTABILE','modalità '+mode
    if not identified:
        return 'NON IDENTIFICATO','misura o disegno non identificato'
    required={'prior_predictive','convergence','ppc','recovery','sensitivity'}
    if row['claim'] in {'M1','H2','H3_location'}:
        required.add('loo_valid')
    failed=[key for key in sorted(required) if gates.get(key) is not True]
    if failed:
        return 'NON RIPORTABILE','; '.join(failed)
    result=probability_class(row['p_support'],row['p_refute'])
    if result=='SUPPORTATA' and row['claim'] in {'M1','H3_location'} and loo_wins is not True:
        return 'INCONCLUSIVA','condizione di vittoria LOO non soddisfatta'
    return result,''

VERDICTS=[]; GATE_TABLE=[]
if MODE!='preflight':
    for row in MAIN_PROBS:
        name,claim=row['model'],row['claim']; spec=SPECS[name]
        loo_claim='H3' if claim=='H3_location' else claim
        gates=dict(prior_predictive=PRIOR_OK.get(name,False),convergence=MAIN_DIAG.get(name,{}).get('ok',False),
                   ppc=PPC_OK.get(name,False),recovery=RECOVERY_OK.get(name,False),sensitivity=SENS_OK.get(claim,False))
        if claim in {'M1','H2','H3_location'}:
            gates['loo_valid']=LOO_VALID.get(loo_claim,False)
        decision,reason=final_decision(row,gates,spec.identified,MODE,LOO_WINS.get(loo_claim))
        VERDICTS.append(dict(**row,verdict=decision,reason=reason,identification_note=spec.reason))
        GATE_TABLE.append(dict(claim=claim,model=name,identified=spec.identified,**gates))
    for name,claim in [('D2_stride','H3a'),('D2_patch','H3b')]:
        if name not in SPECS:
            VERDICTS.append(dict(claim=claim,model=name,parameter='kappa',median=np.nan,low=np.nan,high=np.nan,
                p_support=np.nan,p_refute=np.nan,verdict='NON IDENTIFICATO',
                reason=D2_STATUS[name]['reason'],identification_note=D2_STATUS[name]['reason']))
    VERDICT_TABLE=pd.DataFrame(VERDICTS)
    show(VERDICT_TABLE)
    VERDICT_TABLE.to_csv(OUT/'verdicts.csv',index=False)
    pd.DataFrame(GATE_TABLE).to_csv(OUT/'decision_gates.csv',index=False)
    print('H2: la preferenza LOO è separata dalla ROPE. H3: un solo verdetto congiunto con tutti i controlli.')
    print('Gli eventuali fit D2 non identificati descrivono soltanto il campione selezionato.')
else:
    print('PREFLIGHT terminato: dati e specifiche pronti; nessun posterior, nessun verdetto empirico.')


## Risultati e lettura dei grafici
Il primo grafico controlla la mappatura degli indici di fase usando i dati caricati.
Le figure successive, quando esistono fit, mostrano intervalli posteriori e controlli
predittivi. Un intervallo stretto non scavalca un gate fallito. Nei run smoke tutte
le figure sono prove tecniche; i fit D2 condizionati alla selezione teorica restano
esplicitamente non identificati.


In [ ]:
#@title 17. Figure diagnostiche ed esportazione
fig,ax=plt.subplots(figsize=(8,4))
view=PHASE_AUDIT[PHASE_AUDIT.f_lock.isin([32.,64.,96.,128.])]
palette=['#245B83','#A85D21','#447C61','#84578E']
for (freq,g),color in zip(view.groupby('f_lock'),palette):
    ax.plot(g.phase_idx,np.rad2deg(g.phase_angle),'o-',label=f'{freq:g} Hz',color=color)
ax.set(xlabel='phase_idx storico',ylabel='Fase fisica modulo 360°',title='Lo stesso indice non identifica lo stesso angolo')
ax.legend(title='Frequenza',ncol=2); fig.tight_layout()
fig.savefig(OUT/'phase_mapping.png'); plt.show()

if MODE!='preflight':
    numeric=VERDICT_TABLE[np.isfinite(VERDICT_TABLE['median'])].copy().reset_index(drop=True)
    joint_verdict=VERDICT_TABLE.loc[VERDICT_TABLE.claim=='H3_location','verdict'].iloc[0]
    d1_points=pd.DataFrame([dict(claim='H3_location '+r['parameter'],verdict=joint_verdict,
                                **{k:r[k] for k in ['parameter','median','low','high']}) for r in D1_EFFECTS])
    numeric=pd.concat([numeric,d1_points],ignore_index=True)
    fig,axes=plt.subplots(len(numeric),1,figsize=(9,max(3,1.25*len(numeric))),squeeze=False)
    for i,r in numeric.iterrows():
        ax=axes[i,0]
        ax.errorbar(r['median'],0,xerr=[[r['median']-r.low],[r.high-r['median']]],fmt='o',color='#245B83',capsize=4)
        threshold={'H1_beh':.8,'H1_repr':1.2,'H2':LOG11,'M1':0.,'H3a':1.,'H3b':1.}.get(r.claim)
        if threshold is not None:
            ax.axvline(threshold,color='#555555',linestyle='--')
        ax.set_yticks([]); ax.set_xlabel(r.parameter)
        ax.set_title(f'{r.claim}: {r.verdict} | mediana e intervallo 95%',loc='left',fontsize=10)
    fig.suptitle('Posteriori del run corrente: '+MODE,fontsize=12)
    fig.tight_layout(); fig.savefig(OUT/'posterior_intervals.png'); plt.show()
    rates=PPC_TABLE.groupby(['model','stratum']).ok.mean().unstack()
    fig,ax=plt.subplots(figsize=(max(8,len(rates.columns)*.7),4))
    im=ax.imshow(rates.to_numpy(),vmin=0,vmax=1,cmap='Blues',aspect='auto')
    ax.set_xticks(np.arange(len(rates.columns)),rates.columns,rotation=35,ha='right')
    ax.set_yticks(np.arange(len(rates.index)),rates.index)
    for i in range(len(rates)):
        for j in range(len(rates.columns)):
            v=rates.iloc[i,j]
            ax.text(j,i,'N/A' if pd.isna(v) else f'{v:.0%}',ha='center',va='center',
                    color='white' if pd.notna(v) and v>.65 else '#222222')
    ax.set_title('PPC: quota di controlli coperti, per modello e strato')
    fig.colorbar(im,ax=ax,label='Quota coperta'); fig.tight_layout()
    fig.savefig(OUT/'ppc_coverage.png'); plt.show()
    atomic_json(OUT/'run_summary.json',{'mode':MODE,'code':CODE_SHA256,'sources':SOURCE_HASHES,
        'versions':VERSIONS,'n_fit_calls':len(FIT_LOG),'d2_status':D2_STATUS,
        'empirical_run':MODE=='full','limitations':['B non conserva fase/generatore nella tabella MDL',
        'LOO su sottoinsieme stratificato, condizionato ai gruppi','prior predictive: screening automatico, richiede lettura',
        'D2 storico: misura del fondamentale non indipendente']})
print('Artefatti salvati in:',OUT)


## Cosa si può concludere
- `SUPPORTATA` e `RIFIUTATA` richiedono modalità full, identificazione e tutti i controlli
  previsti per quel modello. Il significato delle soglie è quello dichiarato nelle sezioni precedenti.
- `INCONCLUSIVA` significa che i controlli sono utilizzabili ma la regola non decide.
- `NON RIPORTABILE` indica un controllo mancante/fallito oppure una prova smoke.
- `NON IDENTIFICATO` segnala che la misura o il disegno non permette la conclusione richiesta.

Le correzioni di codice non dimostrano che i dati sostengano H1, H2, H3 o M1. Conservare
il notebook con gli output del run, `data_manifest.json`, i CSV diagnostici e i checkpoint
identificati con hash. Non copiare soltanto la tabella dei verdetti nel report.

**Differenze rispetto all'allegato:** otto fasi fisiche; chiave degli sfondi composta;
regola di inclusione unica; controllo dei contrasti salvati; etichette D1 esatte; distinzione
dei fondamentali D2; raccolta fresca con preflight, manifest completo e nessun riuso silenzioso; nessuna cancellazione
automatica delle raccolte; checkpoint atomici per fit; PPC limitati prima della simulazione;
recovery specifici e ripetuti; LOO appaiato con diagnostiche; sensibilità per tutte le famiglie;
decisione H2 separata dalla preferenza LOO; un solo verdetto congiunto D1; gate espliciti anche D2.